# Tutorial 03 — Time evolution and reconnection rate

The first two tutorials looked at a single timestep. Now we loop over every frame and build two simple time-series diagnostics:

- **psi excursion** — `max(psi) − min(psi)` per frame. A proxy for how much magnetic flux has been displaced by the reconnection.
- **peak |Ez|** — the strongest reconnection electric field anywhere in the domain.

Reading all 51 frames takes a couple of seconds on this dataset. Use `FRAME_STRIDE > 1` if you want it faster on larger runs.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import py3d
from py3d.sub import calc_psi
from data_path import DATA_DIR, PARAM_FILE, NAME_STYLE, require_data_dir

require_data_dir()

In [ ]:
m = py3d.Movie(
    num=0,
    path=DATA_DIR,
    param_file=PARAM_FILE,
    name_style=NAME_STYLE,
    interactive=False,
)

FRAME_STRIDE = 1
frames = list(range(0, m.ntimes, FRAME_STRIDE))
print(f'Will read {len(frames)} of {m.ntimes} frames')

## Build the time axis from sim parameters

Each movie frame is `n_movieout` simulation steps apart, so the time of frame *t* is `t * n_movieout * dt`. We compute it directly from `m.param` rather than relying on `d['tt']` returned by `get_fields` (which is the time axis up to but not including the requested frame).

In [ ]:
dt_movie = m.param['n_movieout'] * m.param['dt']
times = np.array(frames, dtype=float) * dt_movie
print(f'Movie cadence: dt_movie = {dt_movie} 1/Omega_ci')
print(f'Time range:    {times[0]:.2f} to {times[-1]:.2f}')

## Loop over frames

Reading several variables per call is faster than reading them one at a time — the file is opened once per `get_fields`.

In [ ]:
psi_span = np.empty(len(frames))
ez_peak = np.empty(len(frames))

for i, t in enumerate(frames):
    df = m.get_fields('bx by ez', time=t)
    psi = calc_psi(df)
    psi_span[i] = psi.max() - psi.min()
    ez_peak[i] = np.abs(df['ez']).max()
    if (i + 1) % 10 == 0 or i == len(frames) - 1:
        print(f'  frame {t:>3d}  t = {times[i]:6.2f}   '
              f'psi span = {psi_span[i]:6.3f}   '
              f'peak |Ez| = {ez_peak[i]:6.3f}')

## Reconnection rate as the derivative of reconnected flux

`numpy.gradient` gives a centered finite difference at interior points and forward/backward at the ends — a sensible default for noisy time series.

In [ ]:
rec_rate = np.gradient(psi_span, times)

fig, (ax1, ax2) = plt.subplots(
    nrows=2, ncols=1, figsize=(8, 6), sharex=True
)

ax1.plot(times, psi_span, 'k-', lw=1.5, label='psi span')
ax1b = ax1.twinx()
ax1b.plot(times, rec_rate, 'C3--', lw=1.0, label='d(psi span)/dt')
ax1.set_ylabel('psi excursion (max − min)', color='k')
ax1b.set_ylabel('rate of change', color='C3')
ax1b.tick_params(axis='y', colors='C3')
ax1.set_title('Reconnected flux and its growth rate')

ax2.plot(times, ez_peak, 'C0-', lw=1.5)
ax2.set_xlabel('t  (1/Omega_ci)')
ax2.set_ylabel('peak |Ez|')
ax2.set_title('Peak reconnection electric field')

for ax in (ax1, ax2):
    ax.grid(True, alpha=0.3)

fig.tight_layout()

## Summary statistics

In [ ]:
peak_idx = int(np.argmax(rec_rate))
print(f'Peak reconnection rate {rec_rate[peak_idx]:.4f} '
      f'at t = {times[peak_idx]:.2f} (frame {frames[peak_idx]})')
print(f'Peak |Ez| over the run = {ez_peak.max():.4f} '
      f'at t = {times[int(np.argmax(ez_peak))]:.2f}')

## What you just learned

- A frame-by-frame loop is just `for t in range(m.ntimes): df = m.get_fields(...)`. Read multiple variables per call to keep it fast.
- `m.param['n_movieout'] * m.param['dt']` gives the wall-clock cadence between movie frames.
- `numpy.gradient(y, x)` is a clean way to differentiate a time series.

**Next:** [Tutorial 04 — Pressure tensor](04_pressure_tensor.ipynb) loads the pressure tensor and rotates it into the field-aligned frame to compare T∥ and T⊥.